In [1]:
import os
import glob
import numpy as np
import pandas as pd

CONF_DIR = "/Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fmriprep_confounds"
OUT_XLSX = "/Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fmriprep_mean_fd.xlsx"

def get_mean_fd_from_file(tsv_file: str) -> float:
    """
    Read a confounds_timeseries.tsv file and return mean framewise_displacement.
    Returns np.nan if the file cannot be read or the column is missing.
    """
    try:
        df = pd.read_csv(tsv_file, sep="\t")
    except Exception as e:
        print(f"Could not read {tsv_file}: {e}")
        return np.nan

    if "framewise_displacement" not in df.columns:
        print(f"Missing framewise_displacement column in {tsv_file}")
        return np.nan

    fd = pd.to_numeric(df["framewise_displacement"], errors="coerce")
    return fd.mean(skipna=True)

def compute_mean_fd_table(conf_dir: str = CONF_DIR, out_xlsx: str = OUT_XLSX) -> pd.DataFrame:
    """
    Go through copied confounds files in conf_dir and compute mean FD for:
      - task-socialnav
      - task-rest

    Output columns:
      - sub_id: integer subject ID after 'sub-'
      - task_fd: mean FD for task-socialnav
      - rest_fd: mean FD for task-rest

    If a file does not exist, the corresponding value is NaN.
    Saves the table as an Excel file.
    """
    files = sorted(glob.glob(os.path.join(conf_dir, "*confounds_timeseries.tsv")))

    rows = []

    for f in files:
        fname = os.path.basename(f)

        # Expect filenames like:
        # sub-18002_task-rest_desc-confounds_timeseries.tsv
        # sub-18002_task-socialnav_desc-confounds_timeseries.tsv
        parts = fname.split("_")
        sub_part = next((p for p in parts if p.startswith("sub-")), None)
        task_part = next((p for p in parts if p.startswith("task-")), None)

        if sub_part is None or task_part is None:
            print(f"Skipping unrecognized filename: {fname}")
            continue

        sub_id_str = sub_part.replace("sub-", "")
        try:
            sub_id = int(sub_id_str)
        except ValueError:
            print(f"Skipping bad subject ID in filename: {fname}")
            continue

        task_name = task_part.replace("task-", "")
        mean_fd = get_mean_fd_from_file(f)

        rows.append(
            {
                "sub_id": sub_id,
                "task_name": task_name,
                "mean_fd": mean_fd,
            }
        )

    long_df = pd.DataFrame(rows)

    if long_df.empty:
        out_df = pd.DataFrame(columns=["sub_id", "task_fd", "rest_fd"])
        out_df.to_excel(out_xlsx, index=False)
        print(f"No valid files found. Wrote empty file to: {out_xlsx}")
        return out_df

    # If there are multiple files per subject/task, average them
    long_df = (
        long_df.groupby(["sub_id", "task_name"], as_index=False)["mean_fd"]
        .mean()
    )

    # Pivot to wide format
    wide_df = long_df.pivot(index="sub_id", columns="task_name", values="mean_fd").reset_index()

    # Ensure desired output columns exist
    if "socialnav" not in wide_df.columns:
        wide_df["socialnav"] = np.nan
    if "rest" not in wide_df.columns:
        wide_df["rest"] = np.nan

    out_df = wide_df.rename(
        columns={
            "socialnav": "task_fd",
            "rest": "rest_fd",
        }
    )[["sub_id", "task_fd", "rest_fd"]].sort_values("sub_id").reset_index(drop=True)

    out_df.to_excel(out_xlsx, index=False)
    print(f"Saved Excel file to: {out_xlsx}")

    return out_df

if __name__ == "__main__":
    df_out = compute_mean_fd_table()
    print(df_out.head())

Could not read /Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fmriprep_confounds/sub-18009_task-socialnav_desc-confounds_timeseries.tsv: [Errno 60] Operation timed out
Could not read /Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fmriprep_confounds/sub-19005_task-socialnav_desc-confounds_timeseries.tsv: [Errno 60] Operation timed out
Could not read /Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fmriprep_confounds/sub-19042_task-socialnav_desc-confounds_timeseries.tsv: [Errno 60] Operation timed out
Could not read /Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fmriprep_confounds/sub-19048_task-socialnav_desc-confounds_timeseries.tsv: [Errno 60] Operation timed out
Could not read /Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fmriprep_confounds/sub-20009_task-socialnav_desc-confounds_timeseries.tsv: [Errno 60] Operation timed out
Could not read /Users/matty_gee/Desktop/Social/SocialCUD/data/quality-control/fm